In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%reload_ext autoreload

In [0]:
from pyspark.sql.functions import col
from utils.config import catalog_name,schema_name,bronze_table,silver_table,checkpoint_table_silver
from transformations.silver_transformations import merge_silver_table,  create_silver_table, create_dim_publisher, create_dim_employer, create_dim_location
    

In [0]:
catalog=catalog_name
schema=schema_name
bronze_table=bronze_table
silver_table=silver_table
checkpoint_table_silver_path=checkpoint_table_silver

In [0]:
checkpoint_table_silver_path

In [0]:
df=spark.read.table(f"{catalog}.{schema}.{bronze_table}")

In [0]:
bronze_stream = (
    spark.readStream
         .option("readChangeFeed", "true")
         .option("startingVersion", 19)
         .table("job_search_project_catalog.job_search_project_schema.table_bronze_job_search")
         .filter(col("_change_type").isin("insert","update_postimage"))
         )

#merge into silver table
(
bronze_stream.writeStream
    .foreachBatch(merge_silver_table)
    .option("checkpointLocation",checkpoint_table_silver_path)
    .trigger(once=True)
    .start()
)